# Prompting notebook

IDEAS:

- Coger 2 modelos, por ejemplo, Mistral 7B instruct y Qwen del otro notebook

- Probar zero-shot directamente

- Hacerles fine-tuning

- Probar de nuevo zeroshot y ver si hay mejora

La idea en este notebook es probar con una estrategia de zero shot, posiblemente en distintos modelos.

También se puede probar son un prompt más elaborado y otro más sencillito.

In [ ]:
#@title Librerías necesarias
import json
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
!pip install unsloth
import unsloth
from unsloth import FastLanguageModel
import gc
from tqdm import tqdm
import re
import os
from google.colab import drive

In [ ]:
#@title Montar Google Drive
drive.mount('/content/drive')

base_path = '/content/drive/My Drive/'
mistral_path = os.path.join(base_path, 'mistral_results')
qwen_path = os.path.join(base_path, 'qwen_results')




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
SYSTEM_PROMPT = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Solo la letra."""

SYSTEM_PROMPT_BRIEF_REASONING = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Debes responder EXCLUSIVAMENTE con un objeto JSON válido, sin incluir explicaciones previas ni posteriores.
El formato debe ser ESTRICTAMENTE este:
{
  "razonamiento": "Aquí escribes una breve explicación de la respuesta elegida basándote en el texto.",
  "respuesta": "Aquí escribes SOLAMENTE la letra de la opción correcta (A, B, C, D...)."
}"""

CHAIN_OF_THOUGHT = """Eres un profesor experto en resolver exámenes de comprensión lectora en español.
Tu tarea es leer el texto, analizarlo y seleccionar la opción correcta para la pregunta planteada.

Para analizar el texto y responder a las preguntas, debes tener en cuenta estos aspectos:
1. LIMÍTATE AL CONTENIDO DEL TEXTO: Ignora cualquier conocimiento externo o sesgo personal. La validez de una opción depende exclusivamente de la información (explícita o implícita) contenida en el texto.
2. NO TE GUÍES POR LA EXTENSIÓN DE LAS RESPUESTAS: Una opción más detallada o larga no es necesariamente la correcta.
3. ANÁLISIS GLOBAL Y DESCARTE ARGUMENTADO: Evalúa el texto como una unidad (apóyate en marcadores del discurso, tiempos verbales y pronombres) y ten en cuenta que la respuesta correcta casi nunca estará escrita literalmente. Busca equivalencias de significado.
4. EVALÚA TODAS LAS ALTERNATIVAS: debes descartar las opciones incorrectas una a una, aportando argumentos claros de por qué el texto las contradice o no las respalda.

Debes responder EXCLUSIVAMENTE con un objeto JSON válido, sin incluir explicaciones previas ni posteriores.
El formato debe ser ESTRICTAMENTE este:
{
  "razonamiento": "Aquí escribes todo el RAZONAMIENTO necesario para tomar la decisión de cuál es la opción correcta teniendo en cuenta las instrucciones anteriores.",
  "respuesta": "Aquí escribes SOLAMENTE la letra de la opción correcta (A, B, C, D...)."
}
"""

In [ ]:
def load_model_unsloth(model_name, max_seq_length=4096, dtype=None, load_in_4bit=True):
    """
    Carga un modelo y su tokenizador usando Unsloth y lo prepara para inferencia.
    """
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )

    FastLanguageModel.for_inference(model)

    return model, tokenizer

In [ ]:
def load_data():
    """Carga los ficheros JSON para evaluar los modelos."""
    with open('multiple_choice.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    with open('subset_100.json', 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)
    return data, ground_truth

In [ ]:
def filter_questions(data, ground_truth):
    """Filtra las preguntas del subset y prepara la lista de tareas a procesar."""
    tareas = []
    for exam in data['exams']:
        nivel = exam['level']
        for ex_wrapper in exam['exercises']:
            exercise = ex_wrapper['exercise']
            for q in exercise['questions']:
                q_id = q['questionId']
                if q_id in ground_truth:
                    opciones = "\n".join([f"{o['optionId']}) {o['text']}" for o in q['options']])
                    tareas.append({
                        "id": q_id,
                        "nivel": nivel,
                        "contexto": exercise.get('text', ''),
                        "pregunta": q['text'],
                        "opciones": opciones,
                        "real": ground_truth[q_id]
                    })
    return tareas

In [ ]:
def prepare_batch(batch, system_prompt, modo_salida):
    """Construye los mensajes en formato ChatML para un lote de tareas."""
    mensajes = []
    for t in batch:
        base_user = f"Texto:\n{t['contexto']}\n\nPregunta: {t['pregunta']}\nOpciones:\n{t['opciones']}"
        content_user = base_user + "\n\nRespuesta:" if modo_salida == "letra" else base_user

        mensajes.append([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content_user}
        ])
    return mensajes

In [ ]:
def generate_response(model, tokenizer, batch_messages, max_new_tokens):
    """Ejecuta la inferencia pura sobre un lote y devuelve los textos generados."""
    model_inputs = tokenizer.apply_chat_template(
        batch_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        padding=True,
        return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            max_length=None,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    input_len = model_inputs.input_ids.shape[1]
    respuestas_brutas = []
    for output in outputs:
        gen_tokens = output[input_len:]
        texto = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        respuestas_brutas.append(texto)

    return respuestas_brutas

In [ ]:
def process_response(texto_bruto, modo_salida):
    """Extrae la letra (A-D) y la explicación según el formato esperado."""
    prediccion = "N/A"
    explicacion = ""
    error_formato = False

    if modo_salida == "json":
        explicacion = texto_bruto # Guardamos el texto bruto por si falla
        try:
            json_match = re.search(r'\{.*\}', texto_bruto, re.DOTALL)
            if json_match:
                datos = json.loads(json_match.group(0))
                letra_raw = datos.get("respuesta", "").strip().upper()
                match_letra = re.search(r'[A-D]', letra_raw)
                prediccion = match_letra.group(0) if match_letra else "N/A"
                explicacion = datos.get("razonamiento", "")
            else:
                error_formato = True
        except Exception:
            error_formato = True

    elif modo_salida == "letra":
        texto_bruto = texto_bruto.upper()
        match = re.search(r'[A-D]', texto_bruto)
        prediccion = match.group(0) if match else "N/A"
        if not match:
            error_formato = True
    return prediccion, explicacion, error_formato

In [ ]:
def show_results(stats, output_file):
    """Imprime por pantalla el resumen de la evaluación."""
    accuracy_total = (stats["aciertos"] / stats["total"]) * 100 if stats["total"] > 0 else 0
    print("\n" + "="*50)
    print(f"RESULTADOS : {output_file}")
    print("="*50)
    if stats["errores_formato"] > 0:
        print(f"Errores de formato (JSON/Regex fallido): {stats['errores_formato']} de {stats['total']}")
    print(f"Accuracy Global: {accuracy_total:.2f}% ({stats['aciertos']}/{stats['total']})")
    print("-" * 50)
    for nivel, s in sorted(stats["por_nivel"].items()):
        acc_n = (s["aciertos"] / s["total"]) * 100
        print(f"Nivel {nivel}: {acc_n:.2f}% ({s['aciertos']}/{s['total']})")

In [ ]:
def evaluate_model(
    model,
    tokenizer,
    system_prompt,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size=4,
    output_file="resultados.jsonl"
):
    """Función principal que orquesta todo el flujo."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    data, ground_truth = load_data()
    tareas = filter_questions(data, ground_truth)

    resultados_finales = []
    stats = {"total": 0, "aciertos": 0, "errores_formato": 0, "por_nivel": {}}

    if os.path.exists(output_file):
        os.remove(output_file)

    for i in tqdm(range(0, len(tareas), batch_size), desc="Progreso"):
        batch = tareas[i : i + batch_size]

        mensajes = prepare_batch(batch, system_prompt, modo_salida)
        textos_generados = generate_response(model, tokenizer, mensajes, max_new_tokens)

        batch_results = []

        for j, texto_bruto in enumerate(textos_generados):
            prediccion, explicacion, errors = process_response(texto_bruto, modo_salida)

            tarea_actual = batch[j]
            real = tarea_actual["real"]
            nivel = tarea_actual["nivel"]
            es_correcto = (prediccion == real)

            if nivel not in stats["por_nivel"]:
                stats["por_nivel"][nivel] = {"aciertos": 0, "total": 0}

            stats["total"] += 1
            stats["por_nivel"][nivel]["total"] += 1
            if es_correcto:
                stats["aciertos"] += 1
                stats["por_nivel"][nivel]["aciertos"] += 1


            if errors:
                stats["errores_formato"] += 1

            batch_results.append({
                "questionId": tarea_actual["id"],
                "nivel": nivel,
                "pregunta": tarea_actual["pregunta"],
                "respuesta_real": real,
                "prediccion_modelo": prediccion,
                "explicacion": explicacion,
                "error_procesamiento_json": errors,
                "estado": "CORRECTO" if es_correcto else "INCORRECTO"
            })

        with open(output_file, 'a', encoding='utf-8') as f:
          for resultado in batch_results:
              linea_json = json.dumps(resultado, ensure_ascii=False)
              f.write(linea_json + '\n')

    show_results(stats, output_file)

## [Qwen2.5-7B-Instruct](https://huggingface.co/Qwen/Qwen3.5-9B)

In [ ]:
qwen_model, qwen_tokenizer = load_model_unsloth("unsloth/Qwen2.5-7B-Instruct-bnb-4bit")

==((====))==  Unsloth 2026.4.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


### zero-shot

In [ ]:
qwen_results = os.path.join(qwen_path, "simple_qwen_results.json")
evaluate_model(
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    system_prompt=SYSTEM_PROMPT,
    modo_salida="letra",
    max_new_tokens=5,
    batch_size=4,
    output_file=qwen_results
)

Starting inference...


Progreso: 100%|██████████| 25/25 [00:49<00:00,  1.98s/it]


RESULTADOS : /content/drive/My Drive/qwen_results/simple_qwen_results.json
Accuracy Global: 83.00% (83/100)
--------------------------------------------------
Nivel A1: 86.11% (31/36)
Nivel A2: 78.26% (18/23)
Nivel B1: 85.71% (18/21)
Nivel B2: 80.00% (16/20)


### zero-shot con explicación

In [ ]:
qwen_results=os.path.join(qwen_path, "reason_qwen_results.json")
evaluate_model(
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    system_prompt=SYSTEM_PROMPT_BRIEF_REASONING,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size=4,
    output_file=qwen_results
)

Starting inference...


Progreso: 100%|██████████| 25/25 [07:50<00:00, 18.83s/it]


RESULTADOS : /content/drive/My Drive/qwen_results/reason_qwen_results.json
Accuracy Global: 79.00% (79/100)
--------------------------------------------------
Nivel A1: 75.00% (27/36)
Nivel A2: 82.61% (19/23)
Nivel B1: 76.19% (16/21)
Nivel B2: 85.00% (17/20)


### CoT

In [ ]:
qwen_results=os.path.join(qwen_path, "cot_qwen_results.jsonl")
evaluate_model(
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    system_prompt=CHAIN_OF_THOUGHT,
    modo_salida="json",
    max_new_tokens=1500,
    batch_size=6,
    output_file=qwen_results
)

## [Mistral 7B Instruct](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)

In [ ]:
mistral_model, mistral_tokenizer = load_model_unsloth("unsloth/mistral-7b-instruct-v0.3-bnb-4bit")

==((====))==  Unsloth 2026.4.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

### zero-shot

In [ ]:
mistral_results = os.path.join(mistral_path, "simple_zero_shot_mistral.json")
evaluate_model(
    model=mistral_model,
    tokenizer=mistral_tokenizer,
    system_prompt=SYSTEM_PROMPT,
    modo_salida="letra",
    max_new_tokens=5,
    batch_size=4,
    output_file=mistral_results
)

Starting inference...


Progreso:   0%|          | 0/25 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be remove


RESULTADOS : /content/drive/My Drive/mistral_results/simple_zero_shot_mistral.json
Accuracy Global: 73.00% (73/100)
--------------------------------------------------
Nivel A1: 77.78% (28/36)
Nivel A2: 65.22% (15/23)
Nivel B1: 76.19% (16/21)
Nivel B2: 70.00% (14/20)


### zero-shot explicación

In [ ]:
mistral_results = os.path.join(mistral_path, "simple_reasoning_mistral.json")
evaluate_model(
    model=mistral_model,
    tokenizer=mistral_tokenizer,
    system_prompt=SYSTEM_PROMPT_BRIEF_REASONING,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size=4,
    output_file=mistral_results
)

Starting inference...


Progreso: 100%|██████████| 25/25 [08:42<00:00, 20.91s/it]


RESULTADOS : /content/drive/My Drive/mistral_results/simple_reasoning_mistral.json
Accuracy Global: 72.00% (72/100)
--------------------------------------------------
Nivel A1: 80.56% (29/36)
Nivel A2: 60.87% (14/23)
Nivel B1: 66.67% (14/21)
Nivel B2: 75.00% (15/20)


### CoT

In [ ]:
mistral_results = os.path.join(mistral_path, "cot_mistral_results.jsonl")
evaluate_model(
    model=mistral_model,
    tokenizer=mistral_tokenizer,
    system_prompt=CHAIN_OF_THOUGHT,
    modo_salida="json",
    max_new_tokens=1500,
    batch_size=6,
    output_file=mistral_results
)

Progreso: 100%|██████████| 17/17 [15:43<00:00, 55.50s/it]


RESULTADOS : /content/drive/My Drive/mistral_results/cot_mistral_results.jsonl
Accuracy Global: 70.00% (70/100)
--------------------------------------------------
Nivel A1: 69.44% (25/36)
Nivel A2: 69.57% (16/23)
Nivel B1: 66.67% (14/21)
Nivel B2: 75.00% (15/20)


## [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B)

Dado que los resultados con el modelo Qwen2.5-7B-Instruct habían sido bastante buenos, he decidido hacer una tercera prueba con un modelo de la familia Qwen3.5 publicada a comienzos de 2026. Concretamente, he tomado una variante del modelo original que únicamente utilizaba la parte textual del modelo y que estaba cuantizada en 4 bits (https://huggingface.co/techwithsergiu/Qwen3.5-text-9B-bnb-4bit) para no superar los límites de Google Colab.

  


In [ ]:
qwen35_model, qwen35_tokenizer = load_model_unsloth("techwithsergiu/Qwen3.5-text-9B-bnb-4bit")

==((====))==  Unsloth 2026.4.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors:   0%|          | 0.00/7.64G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

techwithsergiu/Qwen3.5-text-9B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


### zero-shot

In [ ]:
qwen35_results_path = os.path.join(qwen_path, "simple_qwen35_results.json")

evaluate_model(
    model=qwen35_model,
    tokenizer=qwen35_tokenizer,
    system_prompt=SYSTEM_PROMPT,
    modo_salida="letra",
    max_new_tokens=5,
    batch_size=4,
    output_file=qwen35_results_path
)

Progreso: 100%|██████████| 25/25 [02:03<00:00,  4.95s/it]



RESULTADOS : /content/drive/My Drive/qwen_results/simple_qwen35_results.json
Accuracy Global: 86.00% (86/100)
--------------------------------------------------
Nivel A1: 91.67% (33/36)
Nivel A2: 78.26% (18/23)
Nivel B1: 85.71% (18/21)
Nivel B2: 85.00% (17/20)


### zero-shot con explicación

In [ ]:
qwen35_results_path = os.path.join(qwen_path, "reason_qwen35_results.json")

evaluate_model(
    model=qwen35_model,
    tokenizer=qwen35_tokenizer,
    system_prompt=SYSTEM_PROMPT_BRIEF_REASONING,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size=4,
    output_file=qwen35_results_path
)

Progreso: 100%|██████████| 25/25 [09:29<00:00, 22.80s/it]


RESULTADOS : /content/drive/My Drive/qwen_results/reason_qwen35_results.json
Accuracy Global: 84.00% (84/100)
--------------------------------------------------
Nivel A1: 86.11% (31/36)
Nivel A2: 82.61% (19/23)
Nivel B1: 80.95% (17/21)
Nivel B2: 85.00% (17/20)


### CoT

In [ ]:
qwen35_results_path = os.path.join(qwen_path, "cot_qwen35_results.json")

evaluate_model(
    model=qwen35_model,
    tokenizer=qwen35_tokenizer,
    system_prompt=CHAIN_OF_THOUGHT,
    modo_salida="json",
    max_new_tokens=1500,
    batch_size=4,
    output_file=qwen35_results_path
)

Progreso: 100%|██████████| 25/25 [41:11<00:00, 98.84s/it] 


RESULTADOS : /content/drive/My Drive/qwen_results/cot_qwen35_results.json
Errores de formato (JSON/Regex fallido): 1 de 100
Accuracy Global: 85.00% (85/100)
--------------------------------------------------
Nivel A1: 86.11% (31/36)
Nivel A2: 82.61% (19/23)
Nivel B1: 85.71% (18/21)
Nivel B2: 85.00% (17/20)


### Auto consistencia

In [ ]:
import collections
import re
import torch
from tqdm import tqdm

def extraer_respuesta_regex(texto):
    """
    Busca la respuesta final en el texto generado por el modelo.
    Devuelve la letra (A, B, C, D) o None si no la encuentra.
    """

    match = re.search(r'"respuesta"\s*:\s*"([A-D])"', texto, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    # Fallback
    match_laxo = re.search(r'([A-D])', texto[::-1])
    if match_laxo:
        return match_laxo.group(1).upper()

    return None

def calcular_voto_mayoritario(votos):
    """
    Recibe una lista de letras (votos) y devuelve la más repetida.
    """
    if not votos:
        return None
    conteo = collections.Counter(votos)
    return conteo.most_common(1)[0][0]

In [ ]:
def generar_secuencias_batch(model, tokenizer, batch_mensajes, n_iterations, temperature, max_new_tokens):
    """
    Prepara el batch, lo pasa por el modelo y genera N secuencias por cada pregunta.
    """
    textos = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch_mensajes]

    inputs = tokenizer(textos, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            max_length=None,
            use_cache=True,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            num_return_sequences=n_iterations,
            pad_token_id=tokenizer.pad_token_id
        )

    prompt_len = inputs['input_ids'].shape[1]
    generations = outputs[:, prompt_len:]
    return tokenizer.batch_decode(generations, skip_special_tokens=True)

In [ ]:
def evaluar_autoconsistencia(model, tokenizer, dataset_mensajes, archivo_salida="resultados_sc.json", batch_size=4, n_iterations=3, temperature=0.7, max_new_tokens=512):
    resultados_finales = []

    if os.path.exists(archivo_salida):
        with open(archivo_salida, 'r', encoding='utf-8') as f:
            try:
                resultados_finales = json.load(f)
            except json.JSONDecodeError:
                print("ERROR: No se puede abrir el fichero")

    # Si hay preguntas ya procesadas no se procesan todas
    inicio_dataset = len(resultados_finales)
    if inicio_dataset >= len(dataset_mensajes):
        return resultados_finales


    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
         tokenizer.pad_token = tokenizer.eos_token

    for i in tqdm(range(inicio_dataset, len(dataset_mensajes), batch_size), desc=f"Progreso (Batch size: {batch_size})"):
        batch_mensajes = dataset_mensajes[i : i + batch_size]

        textos_generados = generar_secuencias_batch(
            model, tokenizer, batch_mensajes, n_iterations, temperature, max_new_tokens
        )

        for idx_pregunta in range(len(batch_mensajes)):
            inicio = idx_pregunta * n_iterations
            fin = inicio + n_iterations
            respuestas_pregunta = textos_generados[inicio:fin]

            votos = []
            respuestas_crudas = []

            for resp_texto in respuestas_pregunta:
                letra = extraer_respuesta_regex(resp_texto)
                if letra:
                    votos.append(letra)
                respuestas_crudas.append((letra, resp_texto))

            ganador = calcular_voto_mayoritario(votos)

            resultados_finales.append({
                "respuesta_mayoritaria": ganador,
                "votos": votos,
                "generaciones": respuestas_crudas
            })

        # Guardar resultados tras cada batch
        with open(archivo_salida, 'w', encoding='utf-8') as f:
            json.dump(resultados_finales, f, ensure_ascii=False, indent=4)

    # Restaurar el tokenizador
    tokenizer.padding_side = original_padding_side

    return resultados_finales

In [ ]:
N_ITERATIONS = 3
TEMPERATURE = 0.7

data, ground_truth = load_data()
tareas = filter_questions(data, ground_truth)

num_ejemplos_prueba = 2
batch_prueba = random.sample(tareas, num_ejemplos_prueba)

mensajes = prepare_batch(tareas, CHAIN_OF_THOUGHT, modo_salida="json")

o_path=os.path.join(qwen_path, "qwen35_self_consistency_results.json")

resultados_sc = evaluar_autoconsistencia(
    model=qwen35_model,
    tokenizer=qwen35_tokenizer,
    dataset_mensajes=mensajes,
    archivo_salida=o_path,
    batch_size=4,
    n_iterations=3,
    temperature=0.7,
    max_new_tokens=1024
)

In [ ]:
stats = {
    "total": 0,
    "aciertos": 0,
    "errores_formato": 0,
    "por_nivel": collections.defaultdict(lambda: {"total": 0, "aciertos": 0})
}

for i, res in enumerate(resultados_sc):
    tarea = tareas[i]
    nivel = tarea.get("nivel", tarea["id"][:2])
    respuesta_real = tarea["real"]
    respuesta_predicha = res["respuesta_mayoritaria"]

    stats["total"] += 1
    stats["por_nivel"][nivel]["total"] += 1

    if respuesta_predicha is None:
        stats["errores_formato"] += 1

    elif respuesta_predicha.upper() == respuesta_real.upper():
        stats["aciertos"] += 1
        stats["por_nivel"][nivel]["aciertos"] += 1

show_results(stats, "evaluacion_qwen_14b_sc.json")


RESULTADOS : evaluacion_qwen_14b_sc.json
Accuracy Global: 86.00% (86/100)
--------------------------------------------------
Nivel A1: 86.11% (31/36)
Nivel A2: 86.96% (20/23)
Nivel B1: 85.71% (18/21)
Nivel B2: 85.00% (17/20)
